# Badge 3: Data Application Builders Workshop: Lesson 2: Getting Started with Streamlit in Snowflake!

The SMOOTHIES database, the Custom Smoothie Order Form app, FRUIT_OPTIONS and the internal stage.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE3_DABW_ADMIN` (ACCOUNTADMIN), `BADGE3_DABW_ROLE` (SYSADMIN) and `BADGE3_DABW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE3_DABW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE3_DABW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE3_DABW_WH"      # stands in for the course's COMPUTE_WH
import ast, io
from snowflake.snowpark.context import get_active_session

CUSTOM = 'SMOOTHIES.PUBLIC."Custom Smoothie Order Form"'
PENDING = 'SMOOTHIES.PUBLIC."Pending Smoothie Orders"'


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result


def write_app(app, app_code):
    """SQL equivalent of editing the app in Snowsight's Streamlit editor: write streamlit_app.py into the app's live version."""
    ast.parse(app_code)  # the code must at least be valid Python
    path = f"snow://streamlit/{app}/versions/live/streamlit_app.py"
    get_active_session().file.put_stream(io.BytesIO(app_code.encode()), path, auto_compress=False, overwrite=True)
    print(f"wrote {len(app_code.splitlines())} lines to {path}")

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: ❓ What is Streamlit in Snowflake?

> **Course:** Check Your Warehouse: you should have a COMPUTE_WH warehouse owned by SYSADMIN (create it if missing; change ownership if ACCOUNTADMIN owns it).

In [ ]:
%%sql -r check_warehouse
-- Derived solution: owner of the stand-in warehouse
use role {{ADMIN_ROLE}};
show warehouses like '{{WAREHOUSE}}';

In [ ]:
w = pdf(check_warehouse).iloc[0]
print('Warehouse', w['name'], 'owner:', w['owner'], '(course: SYSADMIN; stand-in here:', SYSADMIN_ROLE + ')')

> **Course:** Create a Smoothies Database: create a new database named SMOOTHIES, owned by SYSADMIN; the PUBLIC schema also owned by SYSADMIN.

In [ ]:
%%sql -r create_smoothies
-- Derived solution (the course gives no code); created as the SYSADMIN stand-in, so it owns the database and PUBLIC
use role {{SYSADMIN_ROLE}};
create database SMOOTHIES;
use warehouse {{WAREHOUSE}};
use schema SMOOTHIES.PUBLIC;

In [ ]:
%%sql -r check_owners
show databases like 'SMOOTHIES';

In [ ]:
%%sql -r check_public_owner
show schemas like 'PUBLIC' in database SMOOTHIES;

> **Course:** Configure Your User Preferences: set your default role to SYSADMIN and your default warehouse to COMPUTE_WH; tick the resource-monitor email box. (UI: your name → Settings → Preferences.)

In [ ]:
%%sql -r user_defaults
-- SQL equivalent. Commented out: it changes your own user's defaults account-wide, and here the stand-in role and warehouse are not your real SYSADMIN / COMPUTE_WH.
-- alter user identifier($badge_user) set default_role = {{SYSADMIN_ROLE}} default_warehouse = {{WAREHOUSE}};
select 'user defaults left unchanged' as status;

## Page: 👷🏾‍♂️ Let's Make a SiS App

> **Course:** Projects → Streamlit → **+ Streamlit App**; App title **Custom Smoothie Order Form**; app location **SMOOTHIES / PUBLIC**; app warehouse **COMPUTE_WH**; Create (screenshot 2026011315051). DOUBLE-CHECK that you are using SYSADMIN when you create your Streamlit app.

In [ ]:
%%sql -r create_custom_app
-- SQL equivalent of the Create Streamlit App dialog
use role {{SYSADMIN_ROLE}};
create streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"
  runtime_name = 'SYSTEM$WAREHOUSE_RUNTIME'
  query_warehouse = {{WAREHOUSE}}
  title = 'Custom Smoothie Order Form';
alter streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form" add live version from last;

> **Course:** Let's Make Some Edits: collapse the sidebar, delete everything after line 14 of the sample app, change the title (cup_with_straw emoji) and the write text; Run (screenshots 2026011315052, 053).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)
''')

In [ ]:
%%sql -r run_app_1
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"();

## Page: ➕ Adding Interactivity

> **Course:** Paste the Streamlit `st.selectbox` example under your code (screenshots 2026011315071, 072) and Run.

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

import streamlit as st

option = st.selectbox(
    'How would you like to be contacted?',
    ('Email', 'Home phone', 'Mobile phone'))

st.write('You selected:', option)
''')

> **Course:** A Few Tweaks: delete the extra `import streamlit as st`; ask about fruits instead of contact methods (screenshot 2026011315072.png).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

option = st.selectbox(
    'What is your favorite fruit?',
    ('Banana', 'Strawberries', 'Peaches'))

st.write('Your favorite fruit is:', option)
''')

In [ ]:
%%sql -r run_app_2b
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"();

## Page: 🍒 More Fruits to Choose From

> **Course:** Create a FRUIT_OPTIONS table in the PUBLIC schema of SMOOTHIES, owned by SYSADMIN: FRUIT_ID a number, FRUIT_NAME text up to 25 characters (screenshot 2026011315100).

In [ ]:
%%sql -r create_fruit_options
-- Derived solution
use role {{SYSADMIN_ROLE}};
create table SMOOTHIES.PUBLIC.FRUIT_OPTIONS (
  FRUIT_ID number,
  FRUIT_NAME varchar(25)
);

> **Course:** Download the TXT File of Fruit Names & View It: `fruits_available_for_smoothies.txt` (two header rows, `%` delimiter, FRUIT_NAME before FRUIT_ID).

In [ ]:
%%sql -r downloaded_file
-- The learner's Downloads folder stands in as @UTIL_DB.PUBLIC.BADGE3_DOWNLOADS
list @UTIL_DB.PUBLIC.BADGE3_DOWNLOADS;

> **Course:** A FILE FORMAT to Load the Fruit File.

In [ ]:
%%sql -r create_file_format
create file format smoothies.public.two_headerrow_pct_delim
  type = CSV,
  skip_header = 2,
  field_delimiter = '%',
  trim_space = TRUE
;

## Page: 📁 Create & Use an Internal Stage

> **Course:** Create the Internal Stage and Load the File Into It: Create → Stage → Snowflake Managed; name **my_uploaded_files**; SMOOTHIES.PUBLIC; directory table on; client-side encryption (screenshots 2026011315140, 140-1); then **+ Files** and upload the TXT file (141, 142).

In [ ]:
%%sql -r create_internal_stage
-- SQL equivalent of the Create Stage dialog (client-side encryption is the default, SNOWFLAKE_FULL)
use role {{SYSADMIN_ROLE}};
create stage SMOOTHIES.PUBLIC.my_uploaded_files
  directory = (enable = true);

In [ ]:
%%sql -r upload_fruit_file
-- SQL equivalent of **+ Files** → Upload
copy files into @SMOOTHIES.PUBLIC.my_uploaded_files
from @UTIL_DB.PUBLIC.BADGE3_DOWNLOADS
files = ('fruits_available_for_smoothies.txt');
alter stage SMOOTHIES.PUBLIC.my_uploaded_files refresh;
list @SMOOTHIES.PUBLIC.my_uploaded_files;

> **Course:** Query the Not-Yet-Loaded Data Using the File Format (template in the course; screenshot 2026011315145 shows the filled-in version).

In [ ]:
%%sql -r query_staged_file
-- From screenshot 2026011315145
select $1, $2
from @smoothies.public.my_uploaded_files/fruits_available_for_smoothies.txt
(file_format => smoothies.public.two_headerrow_pct_delim);

> **Course:** Check to See if the File WOULD load (the errors are expected: the file's column order differs from the table's; screenshot 2026011315152).

In [ ]:
%%sql -r copy_validation
COPY INTO smoothies.public.fruit_options
FROM @smoothies.public.my_uploaded_files
FILES = ('fruits_available_for_smoothies.txt')
FILE_FORMAT = (FORMAT_NAME = smoothies.public.two_headerrow_pct_delim)
ON_ERROR = ABORT_STATEMENT
VALIDATION_MODE = RETURN_ERRORS
PURGE = TRUE;

## Page: ➕ Bring It All Together

> **Course:** Reorder Columns During the COPY INTO LOAD (screenshots 2026011315170, 171). PURGE = TRUE removes the file from the stage after loading.

In [ ]:
%%sql -r copy_into_reordered
-- From screenshot 2026011315171
COPY INTO smoothies.public.fruit_options
from ( select $2 as FRUIT_ID, $1 as FRUIT_NAME
from @smoothies.public.my_uploaded_files/fruits_available_for_smoothies.txt )
file_format = (format_name = smoothies.public.two_headerrow_pct_delim)
on_error = abort_statement
purge = true;

In [ ]:
%%sql -r check_purge
list @smoothies.public.my_uploaded_files;

> **Course:** Display the Fruit Options List in Your SiS App: add the three course lines (screenshot 2026011315174).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

option = st.selectbox(
    'What is your favorite fruit?',
    ('Banana', 'Strawberries', 'Peaches'))

st.write('Your favorite fruit is:', option)

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options")
st.dataframe(data=my_dataframe, use_container_width=True)
''')

In [ ]:
%%sql -r run_app_2c
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"();

## Page: 🤖 DORA! We Meet Again!

> **Course:** Confirm you have a UTIL_DB owned by SYSADMIN with a PUBLIC schema owned by SYSADMIN; follow the DORA link to create the API integration and GRADER function.

In [ ]:
%%sql -r check_util_db
-- Derived solution: owner of UTIL_DB and its PUBLIC schema
show databases like 'UTIL_DB';

In [ ]:
%%sql -r check_util_db_public
show schemas like 'PUBLIC' in database UTIL_DB;

### 🤖 DORA check DORA_IS_WORKING goes here
The course has the learner run **DORA_IS_WORKING** (the grader responds) at this point.
It is in `dora/dora_tests.ipynb`, section **DORA_IS_WORKING**, as a local check plus the exact DORA code.

## Page: 🤖 DORA DABW001

### 🤖 DORA check DABW001 goes here
The course has the learner run **DABW001** (FRUIT_OPTIONS has 25 rows) at this point.
It is in `dora/dora_tests.ipynb`, section **DABW001**, as a local check plus the exact DORA code.